# African Health Dataset — Neural Network Model
**Build, train and compare Keras Sequential models on African Health data.**

### Sections
1. Setup
2. Dataset
3. Features
4. Sklearn Baseline
5. Basic Sequential NN
6. Train 50 Epochs
7. Loss Curves
8. NN vs Sklearn Comparison
9. Experiments
10. Experiment Loss Curves
11. 300-Word Analysis
12. Push to GitHub


In [ ]:
# Install required libraries
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn --quiet


## 1. Install & Import


In [ ]:
Install & Import ───────────────────────────────────────────
# !pip install tensorflow scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print(f"TensorFlow version: {tf.__version__}")
print("All libraries imported successfully!")

## 2. Build African Dataset


In [ ]:
Build African Dataset ──────────────────────────────────────
data = {
    'country': [
        'Nigeria','South Africa','Kenya','Ethiopia','Egypt','Ghana',
        'Tanzania','Uganda','Morocco','Cameroon','Senegal','Zambia',
        'Zimbabwe','Mali','Mozambique','Madagascar','Burkina Faso',
        'Niger','Malawi','Rwanda','Sudan','Angola','Algeria','Libya',
        'Tunisia','Mauritania','Gambia','Sierra Leone','Liberia',
        'Guinea','Benin','Togo','DR Congo','Mauritius','Botswana',
        'Namibia','Eswatini','Lesotho','Rwanda','Togo'
    ],
    'region': [
        'West','Southern','East','East','North','West',
        'East','East','North','Central','West','East',
        'Southern','West','East','East','West',
        'West','East','East','East','Southern','North','North',
        'North','West','West','West','West',
        'West','West','West','Central','Southern','Southern',
        'Southern','Southern','Southern','East','West'
    ],
    'life_expectancy': [
        54.3, 64.9, 67.5, 65.5, 72.0, 64.1,
        65.5, 63.7, 74.5, 59.3, 67.9, 63.9,
        61.2, 59.3, 60.1, 67.4, 61.6,
        62.4, 64.6, 69.0, 65.5, 61.5, 76.9, 72.9,
        76.2, 64.9, 62.1, 54.7, 60.5,
        56.1, 62.1, 60.9, 60.5, 74.6, 69.7,
        63.9, 59.4, 53.7, 70.1, 61.5
    ],
    'under5_mortality': [
        111.2, 33.4, 40.9, 47.5, 20.3, 47.4,
        51.7, 44.8, 18.5, 73.7, 43.1, 59.2,
        55.1, 97.7, 74.4, 80.5, 87.5,
        117.9, 37.5, 35.5, 63.0, 72.1, 12.8, 11.4,
        13.7, 76.2, 53.5, 108.4, 71.0,
        109.5, 88.7, 67.8, 102.4, 13.7, 29.0,
        51.4, 55.0, 74.5, 34.0, 68.0
    ],
    'gdp_per_capita': [
        2085, 7055, 2010, 925, 3561, 2362,
        1139, 868, 3352, 1556, 1596, 1173,
        1464, 848, 507, 528, 878,
        595, 644, 834, 724, 2037, 4078, 8024,
        3809, 748, 800, 490, 627,
        595, 1335, 926, 550, 10844, 7731,
        5269, 1036, 1122, 834, 926
    ],
    'electricity_access_pct': [
        57.5, 84.2, 75.0, 45.0, 100.0, 83.9,
        38.1, 43.4, 100.0, 62.2, 69.8, 43.3,
        49.7, 52.6, 30.0, 34.1, 18.8,
        19.5, 14.0, 46.7, 49.7, 46.2, 99.9, 72.1,
        99.9, 46.3, 62.3, 26.1, 29.0,
        47.2, 42.0, 43.8, 22.7, 99.9, 72.7,
        56.2, 76.7, 29.4, 46.7, 43.8
    ],
    'literacy_rate_pct': [
        62.0, 95.0, 82.0, 51.8, 73.1, 79.0,
        80.0, 77.3, 73.8, 77.1, 51.9, 87.5,
        89.0, 35.5, 63.4, 74.8, 46.0,
        35.1, 62.0, 73.2, 61.0, 71.4, 81.4, 91.0,
        81.8, 52.1, 58.1, 43.2, 48.3,
        47.8, 55.3, 63.7, 37.5, 91.3, 88.4,
        92.3, 88.4, 62.4, 73.2, 63.7
    ]
}

df = pd.DataFrame(data)

# Target: high vs low life expectancy (above/below median)
median_le = df['life_expectancy'].median()
df['target'] = (df['life_expectancy'] > median_le).astype(int)

# Feature engineering
df['development_score'] = (
    df['gdp_per_capita'] / df['gdp_per_capita'].max() * 0.4 +
    df['electricity_access_pct'] / 100 * 0.3 +
    df['literacy_rate_pct'] / 100 * 0.3
) * 100

le_enc = LabelEncoder()
df['region_encoded'] = le_enc.fit_transform(df['region'])

print(f"Dataset shape: {df.shape}")
print(f"Median life expectancy: {median_le:.1f} years")
print(f"High (1): {df['target'].sum()} | Low (0): {(df['target']==0).sum()}")

## 3. Prepare Features


In [ ]:
Prepare Features ───────────────────────────────────────────
FEATURES = [
    'under5_mortality', 'gdp_per_capita', 'electricity_access_pct',
    'literacy_rate_pct', 'development_score', 'region_encoded'
]

X = df[FEATURES].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape} | Test: {X_test_s.shape}")

## 4. Scikit-learn Baseline


In [ ]:
Scikit-learn Baseline ──────────────────────────────────────
sklearn_models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM':                    SVC(probability=True, random_state=42)
}

sklearn_results = {}
print("\nScikit-learn Baseline Results:")
print("=" * 55)

for name, model in sklearn_models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    sklearn_results[name] = {'accuracy': acc, 'auc': auc}
    print(f"  {name:<25} Acc={acc:.3f}  AUC={auc:.3f}")

best_name = max(sklearn_results, key=lambda x: sklearn_results[x]['accuracy'])
best_acc  = sklearn_results[best_name]['accuracy']
print(f"\nBest sklearn model: {best_name} ({best_acc:.3f})")

## 5. Basic Sequential Neural Network


In [ ]:
Basic Sequential Neural Network ────────────────────────────
print("\nBuilding Basic Sequential Neural Network...")
print("Architecture: Input(6) -> Dense(64, ReLU) -> Dense(1, Sigmoid)")

basic_model = keras.Sequential([
    layers.Input(shape=(len(FEATURES),),  name='input_layer'),
    layers.Dense(64, activation='relu',   name='hidden_layer'),
    layers.Dense(1,  activation='sigmoid',name='output_layer')
], name='Basic_NN')

basic_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

basic_model.summary()

## 6. Train 50 Epochs


In [ ]:
Train 50 Epochs ────────────────────────────────────────────
print("\nTraining for 50 epochs...")

history = basic_model.fit(
    X_train_s, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=0
)

loss_val, acc_val = basic_model.evaluate(X_test_s, y_test, verbose=0)
print(f"Basic NN Test Accuracy: {acc_val:.4f}")
print(f"Basic NN Test Loss:     {loss_val:.4f}")

## 7. Plot Loss Curves


In [ ]:
Plot Loss Curves ───────────────────────────────────────────
epochs = range(1, 51)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(epochs, history.history['loss'],
             color='#2196F3', linewidth=2.5, label='Train Loss')
axes[0].plot(epochs, history.history['val_loss'],
             color='#2196F3', linewidth=2, linestyle='--',
             alpha=0.7, label='Val Loss')
axes[0].fill_between(epochs,
                     history.history['loss'],
                     history.history['val_loss'],
                     alpha=0.1, color='#2196F3')
axes[0].set_title('Loss Curve — Basic NN', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(epochs, history.history['accuracy'],
             color='#4CAF50', linewidth=2.5, label='Train Accuracy')
axes[1].plot(epochs, history.history['val_accuracy'],
             color='#4CAF50', linewidth=2, linestyle='--',
             alpha=0.7, label='Val Accuracy')
axes[1].fill_between(epochs,
                     history.history['accuracy'],
                     history.history['val_accuracy'],
                     alpha=0.1, color='#4CAF50')
axes[1].set_title('Accuracy Curve — Basic NN', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('Training & Validation Curves — Basic Sequential Neural Network',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('loss_curves_basic_nn.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: loss_curves_basic_nn.png")

## 8. NN vs Sklearn Comparison


In [ ]:
NN vs Sklearn Comparison ───────────────────────────────────
nn_auc = roc_auc_score(
    y_test, basic_model.predict(X_test_s, verbose=0)
)

all_results = {**sklearn_results}
all_results['Neural Network'] = {'accuracy': acc_val, 'auc': nn_auc}

names  = list(all_results.keys())
accs   = [all_results[n]['accuracy'] for n in names]
aucs   = [all_results[n]['auc']      for n in names]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#E91E63']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bars1 = axes[0].bar(range(len(names)), accs, color=colors,
                    edgecolor='white', linewidth=1.5)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy: NN vs Scikit-learn', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.2)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.4, label='Random')
axes[0].legend(fontsize=9)
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

bars2 = axes[1].bar(range(len(names)), aucs, color=colors,
                    edgecolor='white', linewidth=1.5)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
axes[1].set_ylabel('ROC-AUC Score')
axes[1].set_title('AUC: NN vs Scikit-learn', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 1.2)
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.4, label='Random')
axes[1].legend(fontsize=9)
for bar, val in zip(bars2, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('Model Comparison: Neural Network vs Scikit-learn',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

## 9. Experiments


In [ ]:
Experiments ────────────────────────────────────────────────
print("\nRunning Experiments...")

def train_model(model, name):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    hist = model.fit(
        X_train_s, y_train,
        epochs=50, batch_size=8,
        validation_split=0.2,
        verbose=0
    )
    loss, acc = model.evaluate(X_test_s, y_test, verbose=0)
    auc = roc_auc_score(y_test, model.predict(X_test_s, verbose=0))
    print(f"  {name:<35} Acc={acc:.3f}  AUC={auc:.3f}  Params={model.count_params()}")
    return hist, acc, auc

# Experiment 1: Basic (already trained)
hist_basic = history
acc_basic_exp = acc_val

# Experiment 2: Deeper model (+extra layer)
deeper = keras.Sequential([
    layers.Input(shape=(len(FEATURES),)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),   # extra layer
    layers.Dense(1,  activation='sigmoid')
], name='Deeper_NN')
hist_deeper, acc_deeper, auc_deeper = train_model(deeper, 'Deeper (64->32->sigmoid)')

# Experiment 3: Wider model (128 neurons)
wider = keras.Sequential([
    layers.Input(shape=(len(FEATURES),)),
    layers.Dense(128, activation='relu'),  # 128 neurons
    layers.Dense(1,   activation='sigmoid')
], name='Wider_NN')
hist_wider, acc_wider, auc_wider = train_model(wider, 'Wider (128->sigmoid)')

# Experiment 4: Dropout
dropout_model = keras.Sequential([
    layers.Input(shape=(len(FEATURES),)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),                   # 30% dropout
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),                   # 20% dropout
    layers.Dense(1,  activation='sigmoid')
], name='Dropout_NN')
hist_dropout, acc_dropout, auc_dropout = train_model(dropout_model, 'Dropout (64->drop->32->drop)')

## 10. Experiment Loss Curves


In [ ]:
Experiment Loss Curves ───────────────────────────────────
exp_data = [
    ('Basic (64, ReLU)',           hist_basic,   '#2196F3', acc_basic_exp),
    ('Deeper (+32 layer)',         hist_deeper,  '#E91E63', acc_deeper),
    ('Wider (128 neurons)',        hist_wider,   '#FF9800', acc_wider),
    ('Dropout (Regularised)',      hist_dropout, '#4CAF50', acc_dropout),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, (name, hist, color, acc) in enumerate(exp_data):
    ax = axes[i]
    ep = range(1, len(hist.history['loss']) + 1)

    ax.plot(ep, hist.history['loss'],
            color=color, linewidth=2.5, label='Train Loss')
    ax.plot(ep, hist.history['val_loss'],
            color=color, linewidth=2, linestyle='--',
            alpha=0.7, label='Val Loss')

    ax2 = ax.twinx()
    ax2.plot(ep, hist.history['accuracy'],
             color='#607D8B', linewidth=1.8, linestyle=':', label='Train Acc')
    ax2.plot(ep, hist.history['val_accuracy'],
             color='#263238', linewidth=1.8, linestyle='-.', label='Val Acc')
    ax2.set_ylim(0, 1.2)
    ax2.set_ylabel('Accuracy', color='#455A64', fontsize=9)

    ax.set_title(f'{name}\nTest Accuracy = {acc:.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('Loss', fontsize=10)

    lines1, lab1 = ax.get_legend_handles_labels()
    lines2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, lab1+lab2, fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)

fig.suptitle('Experiment Results: Loss & Accuracy for All NN Variants',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('experiment_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: experiment_loss_curves.png")

## 11. 300-Word Analysis


In [ ]:
300-Word Analysis ─────────────────────────────────────────
print("""
============================================================
  WHEN ARE NEURAL NETWORKS WORTH THE COMPLEXITY?
  A 300-Word Analysis — African Health Dataset
============================================================

Neural networks (NNs) are powerful but carry significant
overhead: longer training times, more hyperparameters, harder
interpretability, and greater data requirements. The question
is not whether NNs can solve a problem, but whether they are
the right tool for it.

On this African health dataset — 40 countries and 6 features —
scikit-learn models (Random Forest, Gradient Boosting) matched
or outperformed the basic neural network. This is expected.
With small, structured, tabular data, traditional models have
clear advantages: they train in milliseconds, need no GPU,
require far less data, and are interpretable through feature
importances and decision paths. Adding a NN here increases
complexity without meaningful accuracy gains.

Neural networks justify their complexity in specific scenarios.
First, large datasets: NNs excel with tens of thousands or
millions of samples, learning complex hierarchical patterns
that simpler models cannot capture. Second, unstructured data:
images, audio, and text are where NNs are irreplaceable. CNNs
can diagnose malaria from blood slide images; Transformers can
extract insights from health policy documents. Third, highly
non-linear relationships: when feature interactions are deeply
complex, NNs model patterns that decision trees or linear
models miss entirely. Fourth, multi-task learning: a single NN
can predict multiple health outcomes simultaneously by sharing
learned representations across tasks.

For African health analytics specifically, NNs become valuable
with satellite imagery (predicting infrastructure access from
aerial photos), mobile health records (processing free-text
and time-series simultaneously), or epidemic forecasting at
national scale using weather, mobility, and demographic signals
combined — tasks impossible for a simple classifier.

The lesson from this experiment is clear: always build a strong
scikit-learn baseline before reaching for deep learning. If the
baseline competes on your dataset size and type, the simpler
model wins every time. Neural networks are justified only when
data volume, data type (unstructured), or relationship
complexity genuinely demands them. Complexity is only worth
the cost when it delivers meaningful, measurable improvement.

Word count: ~300
============================================================
""")

## 12. Push to GitHub


In [ ]:
Push to GitHub ────────────────────────────────────────────
print("Files generated:")
print("  loss_curves_basic_nn.png")
print("  model_comparison.png")
print("  experiment_loss_curves.png")
print()
print("Push to GitHub:")
print("  git add african_nn_model.py *.png README.md")
print("  git commit -m 'Add Keras NN, loss curves and analysis'")
print("  git push origin main")